# Step 1: Collect Activation Differences

This notebook demonstrates how to collect activation differences between Gemma 2 base and IT models.

In [ ]:
import sys
sys.path.append('..')

import torch
from datasets import load_dataset
from src.models import load_gemma_models
from src.utils import DiffActivationCollector
import yaml

## Load Configuration

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"Target layer: {config['models']['target_layer']}")
print(f"SAE n_latents: {config['sae']['n_latents']}")

## Load Models

In [ ]:
print("Loading Gemma 2 models...")
models = load_gemma_models('../configs/config.yaml')
print(models)

## Load Dataset

In [ ]:
# Load a sample dataset (you can replace with your own)
dataset = load_dataset(
    config['data']['dataset_name'],
    split='train[:1000]'  # Use first 1000 samples for quick testing
)

# Extract text prompts
# Adjust this based on your dataset structure
prompts = [item['messages'][0]['content'] for item in dataset if item['messages']]

print(f"Loaded {len(prompts)} prompts")
print(f"\nExample prompt: {prompts[0]}")

## Collect Activation Differences

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

collector = DiffActivationCollector(
    models.base_model,
    models.instruct_model,
    models.tokenizer,
    config['models']['target_layer'],
    device
)

print(f"Collecting activations from layer {config['models']['target_layer']}...")

In [ ]:
# Collect activations
activations = collector.collect_diff_activations(
    prompts,
    batch_size=config['data']['batch_size'],
    max_length=config['data']['max_seq_length'],
    return_individual=True  # Also save base and IT activations
)

print("\nActivation shapes:")
for key, value in activations.items():
    print(f"  {key}: {value.shape}")

## Save Activations

In [ ]:
save_path = '../data/activations/diff_activations.pkl'

collector.save_activations(activations, save_path)

print(f"\nActivations saved to {save_path}")
print(f"Ready for SAE training!")

## Quick Statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Flatten for statistics
diff_flat = activations['diff'].reshape(-1, activations['diff'].shape[-1])

print("Diff Activation Statistics:")
print(f"  Mean: {diff_flat.mean():.6f}")
print(f"  Std: {diff_flat.std():.6f}")
print(f"  Min: {diff_flat.min():.6f}")
print(f"  Max: {diff_flat.max():.6f}")

# Plot histogram
plt.figure(figsize=(10, 5))
plt.hist(diff_flat[:, 0].numpy(), bins=50, alpha=0.7)
plt.xlabel('Activation Difference')
plt.ylabel('Frequency')
plt.title('Distribution of Activation Differences (First Dimension)')
plt.show()